In [ ]:
%reload_ext autoreload
%autoreload 2

import os
import numpy as np
import matplotlib.pyplot as plt

from tqdm import tqdm
from yacs.config import load_cfg
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score
# from sklearn.metrics import mean_squared_error
# from sklearn.metrics import f1_score

from bp_models import MODELS
from bp_datasets import DATASETS
from bp_utils import load_checkpoint, count_model_params


RUN_NAME = "20240227_234656"    # <-- update run name
DATA_ROOT = os.path.join("..")  # <-- update dataset root directory

CHECKPOINT_PATH = os.path.join("..", "runs", RUN_NAME, "checkpoints", "chkpt_best.pt")
# CHECKPOINT_PATH = os.path.join("..", "runs", RUN_NAME, "checkpoints", "chkpt_ep100.pt")
BATCH_SIZE = 32
DEVICE = "cuda"

with open(os.path.join("..", "runs", RUN_NAME, "config.yml")) as f:
  cfg = load_cfg(f.read())

print("Model:", cfg.MODEL.NAME)
print("Dataset:", cfg.DATASET.NAME)

In [ ]:
# load model
model = MODELS[cfg.MODEL.NAME](cfg).to(DEVICE)
_ = load_checkpoint(
    p=CHECKPOINT_PATH,
    model=model,
    device=DEVICE,
)
num_params = count_model_params(model)
print("The model has {:,} parameters.".format(num_params))

In [ ]:
# load dataset
ds = DATASETS[cfg['DATASET']['NAME']](DATA_ROOT, "eval")
data_loader = DataLoader(dataset=ds, batch_size=BATCH_SIZE, shuffle=True)
print("Testing dataset has {:,} samples.".format(len(ds)))

In [ ]:
# example test loop
model.eval()

predictions = []
groud_truth = []

for inputs, targets in tqdm(data_loader, desc="Evaluation"):
    inputs = inputs.to(DEVICE)
    targets = targets.to(DEVICE)

    _, outputs = model(inputs)

    predictions.append(outputs.detach().cpu().numpy())
    groud_truth.append(targets.detach().cpu().numpy())

# assuming the first axis is the batch size
predictions = np.concatenate(predictions, axis=0)
groud_truth = np.concatenate(groud_truth, axis=0)


In [ ]:
# calculate evaluation metric
accuracy_score(groud_truth, predictions)

In [ ]:
# additional qualitative tests
# ...
